In [1]:
import pandas as pd
import json

In [2]:
import glob
files = glob.glob("../data/external/*V2*instruct*.json")
files

['../data/external/first_100_failing_examples_without_docstrings_base_model_og_prompt_V2_with_instruct_response.json',
 '../data/external/first_100_passing_examples_without_docstrings_base_model_og_prompt_V2_with_instruct_response.json',
 '../data/external/first_100_selected_examples_without_docstrings_base_model_og_prompt_V2_instruct_comparison_with_position_info.json',
 '../data/external/first_100_selected_examples_without_docstrings_base_model_og_prompt_V2_instruct_comparison_with_position_info_post_process.json']

In [3]:
def create_input_prompt_prefix(task_prompt, test_list, mode = "base"):
    prompt = (
            "You are an expert Python programmer, and here is your task: "
            f"{task_prompt} Your code should pass these tests:\n\n"
            + "\n".join(test_list) + "\nWrite your code below starting with \"```python\" and ending with \"```\".\n```python\n"
        )
    if mode == "instruct":
        prompt = (
            "You are an expert Python programmer, and here is your task: "
            f"{task_prompt} Your code should pass these tests:\n\n"
            + "\n".join(test_list) + "\nWrite your code, without docstrings, below starting with \"```python\" and ending with \"```\".\n```python\n"
        )
    
    return prompt

In [4]:
def filter_base_output(base_output):
    if "return 0" in base_output:
        return False
    if base_output.strip() == "":
        return False
    if '"""' in base_output or "'''" in base_output:
        return False
    return True

In [5]:
import ast

class Normalizer(ast.NodeTransformer):
    def __init__(self):
        self.var_map = {}
        self.func_map = {}
        self.class_map = {}
        self.counter = 0

    def _rename(self, name, mapping):
        if name not in mapping:
            mapping[name] = f"id_{len(mapping)}"
        return mapping[name]

    def visit_Name(self, node):
        node.id = self._rename(node.id, self.var_map)
        return node

    def visit_arg(self, node):
        node.arg = self._rename(node.arg, self.var_map)
        return node

    def visit_FunctionDef(self, node):
        node.name = self._rename(node.name, self.func_map)
        self.generic_visit(node)
        return node

    def visit_ClassDef(self, node):
        node.name = self._rename(node.name, self.class_map)
        self.generic_visit(node)
        return node

def normalize(code):
    try:
        tree = ast.parse(code)
        normalized = Normalizer().visit(tree)
        return ast.dump(normalized, annotate_fields=True, include_attributes=False)
    except:
        return None

In [6]:
def check_if_abstract_tree_same(base_response, instruct_response):
    return normalize(base_response) == normalize(instruct_response)

In [7]:
import difflib

def similarity(code1, code2):
    return difflib.SequenceMatcher(None, code1, code2).ratio()

In [8]:
import astpretty

def pretty_ast(code):    
    tree = Normalizer().visit(ast.parse(code))
    astpretty.pprint(tree)


In [9]:
sample_base_response = "def sum_series(n):\n    return n - 2 * (n // 2)"
sample_instruct_response = "def sum_series(n):\n  total = 0\n  for i in range(n // 2):\n    total += n - 2 * i\n  return total\n"

In [10]:
ast_base = normalize(sample_base_response)

In [11]:
pretty_ast(sample_base_response)

Module(
    body=[
        FunctionDef(
            lineno=1,
            col_offset=0,
            end_lineno=2,
            end_col_offset=27,
            name='id_0',
            args=arguments(
                posonlyargs=[],
                args=[arg(lineno=1, col_offset=15, end_lineno=1, end_col_offset=16, arg='id_0', annotation=None, type_comment=None)],
                vararg=None,
                kwonlyargs=[],
                kw_defaults=[],
                kwarg=None,
                defaults=[],
            ),
            body=[
                Return(
                    lineno=2,
                    col_offset=4,
                    end_lineno=2,
                    end_col_offset=27,
                    value=BinOp(
                        lineno=2,
                        col_offset=11,
                        end_lineno=2,
                        end_col_offset=27,
                        left=Name(lineno=2, col_offset=11, end_lineno=2, end_col_offset=12, id='id_0', ct

In [12]:
pretty_ast(sample_instruct_response)

Module(
    body=[
        FunctionDef(
            lineno=1,
            col_offset=0,
            end_lineno=5,
            end_col_offset=14,
            name='id_0',
            args=arguments(
                posonlyargs=[],
                args=[arg(lineno=1, col_offset=15, end_lineno=1, end_col_offset=16, arg='id_0', annotation=None, type_comment=None)],
                vararg=None,
                kwonlyargs=[],
                kw_defaults=[],
                kwarg=None,
                defaults=[],
            ),
            body=[
                Assign(
                    lineno=2,
                    col_offset=2,
                    end_lineno=2,
                    end_col_offset=11,
                    targets=[Name(lineno=2, col_offset=2, end_lineno=2, end_col_offset=7, id='id_1', ctx=Store())],
                    value=Constant(lineno=2, col_offset=10, end_lineno=2, end_col_offset=11, value=0, kind=None),
                    type_comment=None,
                ),
    

In [13]:
def _extract_code_part(input_prefix_text, suffix_text):
    full_text = input_prefix_text + suffix_text
    return "def" + full_text.split("```python\n")[-1].split("def")[1]

def check_if_answers_are_different_after_signature(entry, threshold = 0.95):
    instruct_output = entry["instruct_code"]
    base_output = entry["model_output"]
    task = entry["prompt"]
    test_list = entry["test_list"]
    base_prefix = create_input_prompt_prefix(task, test_list, mode = "base")
    instruct_prefix = create_input_prompt_prefix(task, test_list, mode = "instruct")
    base_code = _extract_code_part(base_prefix, base_output)
    instruct_code = _extract_code_part(instruct_prefix, instruct_output)
    normalized_base_code = normalize(base_code)
    normalized_instruct_code = normalize(instruct_code)
    if normalized_base_code != None and normalized_instruct_code != None:
        return normalized_base_code != normalized_instruct_code
    else:
        return similarity(base_code, instruct_code) < threshold

In [14]:
failing_file = files[0]
passing_file = files[1]

all_task_list = []

for file in [failing_file, passing_file]:
    with open(file, "r") as f:
        data = json.load(f)
    all_task_list.extend(data)

len(all_task_list), all_task_list[0].keys()

(100,
 dict_keys(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list', 'model_output', 'instruct_code']))

In [15]:
filtered_task_list = [task for task in all_task_list if filter_base_output(task['model_output'])]
len(filtered_task_list), filtered_task_list[0].keys()

(89,
 dict_keys(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list', 'model_output', 'instruct_code']))

In [16]:
different_answer_list = [task for task in filtered_task_list if check_if_answers_are_different_after_signature(task)]
same_answer_list = [task for task in filtered_task_list if not check_if_answers_are_different_after_signature(task)]

len(different_answer_list), different_answer_list[0].keys()

(59,
 dict_keys(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list', 'model_output', 'instruct_code']))

In [17]:
same_answer_list[23]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 262,
 'prompt': 'Write a function that takes in a list and an integer L and splits the given list into two parts where the length of the first part of the list is L, and returns the resulting lists in a tuple.',
 'code': 'def split_two_parts(list1, L):\n    return list1[:L], list1[L:]',
 'test_imports': [],
 'test_list': ['assert split_two_parts([1,1,2,3,4,4,5,1],3)==([1, 1, 2], [3, 4, 4, 5, 1])',
  "assert split_two_parts(['a', 'b', 'c', 'd'],2)==(['a', 'b'], ['c', 'd'])",
  "assert split_two_parts(['p', 'y', 't', 'h', 'o', 'n'],4)==(['p', 'y', 't', 'h'], ['o', 'n'])"],
 'model_output': 'def split_two_parts(list, L):\n    return (list[:L], list[L:])',
 'instruct_code': 'def split_two_parts(lst, L):\n    return (lst[:L], lst[L:])\n'}

In [18]:
same_answer_list[9]

{'source_file': "Mike's Copy of Benchmark Questions Verification V2.ipynb",
 'task_id': 66,
 'prompt': 'Write a python function to count the number of positive numbers in a list.',
 'code': 'def pos_count(list):\n  pos_count= 0\n  for num in list: \n    if num >= 0: \n      pos_count += 1\n  return pos_count ',
 'test_imports': [],
 'test_list': ['assert pos_count([1,-2,3,-4]) == 2',
  'assert pos_count([3,4,5,-1]) == 3',
  'assert pos_count([1,2,3,4]) == 4'],
 'model_output': 'def pos_count(lst):\n    count = 0\n    for i in lst:\n        if i > 0:\n            count += 1\n    return count',
 'instruct_code': 'def pos_count(nums):\n  count = 0\n  for num in nums:\n    if num > 0:\n      count += 1\n  return count\n'}

In [19]:
different_answer_list[9]

{'source_file': "Mike's Copy of Benchmark Questions Verification V2.ipynb",
 'task_id': 132,
 'prompt': 'Write a function to convert a tuple to a string.',
 'code': "def tup_string(tup1):\n  str =  ''.join(tup1)\n  return str",
 'test_imports': [],
 'test_list': ['assert tup_string((\'e\', \'x\', \'e\', \'r\', \'c\', \'i\', \'s\', \'e\', \'s\'))==("exercises")',
  'assert tup_string((\'p\',\'y\',\'t\',\'h\',\'o\',\'n\'))==("python")',
  'assert tup_string((\'p\',\'r\',\'o\',\'g\',\'r\',\'a\',\'m\'))==("program")'],
 'model_output': 'def tup_string(tup):\n    return str(tup)',
 'instruct_code': "def tup_string(tup):\n    return ''.join(tup)\n"}

In [20]:
import ast
import inspect
import textwrap

def extract_function_body(code: str, func_name: str = None) -> str:
    """Return the source of the function body (as a string), excluding signature."""
    tree = ast.parse(code)

    # Find the first function if name not specified
    func = None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            if func_name is None or node.name == func_name:
                func = node
                break

    if func is None:
        raise ValueError("No function definition found.")

    # Get the full source of the function
    src = code.splitlines()

    # Extract body line numbers
    start = func.body[0].lineno - 1  # body starts here
    end = func.end_lineno            # end of function

    # Extract just the body text
    body = "\n".join(src[start:end])
    return textwrap.dedent(body)


def first_difference(a: str, b: str):
    """Return index of first differing character, or None."""
    for i, (ca, cb) in enumerate(zip(a, b)):
        if ca != cb:
            return i
    if len(a) != len(b):
        return min(len(a), len(b))
    return None


# ---------- Example usage ----------

code1 = """
def f(x, y):
    a = x + y
    b = a * 2
    return b
"""

code2 = """
def f(x, y):
    a = x - y
    b = a * 2
    return b
"""

body1 = extract_function_body(code1)
body2 = extract_function_body(code2)

idx = first_difference(body1, body2)
print("First differing index:", idx)
if idx is not None:
    print("Code1:", body1[idx:idx+20])
    print("Code2:", body2[idx:idx+20])

First differing index: 6
Code1: + y
b = a * 2
return
Code2: - y
b = a * 2
return


In [21]:
base_code, instruct_code = different_answer_list[9]["model_output"], different_answer_list[9]["instruct_code"]
base_code, instruct_code

('def tup_string(tup):\n    return str(tup)',
 "def tup_string(tup):\n    return ''.join(tup)\n")

In [22]:
# ---------- Example usage ----------
def find_first_difference(code1, code2, verbose = False):
    body1 = extract_function_body(code1)
    body2 = extract_function_body(code2)

    idx = first_difference(body1, body2)
    if verbose:
        print("First differing index:", idx)
        if idx is not None:
            print("Code1:", body1[idx:idx+20])
            print("Code2:", body2[idx:idx+20])

    diff_code1 = body1[idx:idx+10]
    diff_code2 = body2[idx:idx+10]

    return idx, diff_code1, diff_code2

find_first_difference(base_code, instruct_code, True)

First differing index: 7
Code1: str(tup)
Code2: ''.join(tup)


(7, 'str(tup)', "''.join(tu")

In [23]:
alt_base_code = "def tup_string(b):\n    return str(b)"
find_first_difference(alt_base_code, instruct_code, verbose = True)

First differing index: 7
Code1: str(b)
Code2: ''.join(tup)


(7, 'str(b)', "''.join(tu")

In [40]:
base_output = same_answer_list[9]['model_output']
instruct_output = same_answer_list[9]['instruct_code']
base_output, instruct_output

('def pos_count(lst):\n    count = 0\n    for i in lst:\n        if i > 0:\n            count += 1\n    return count',
 'def pos_count(nums):\n  count = 0\n  for num in nums:\n    if num > 0:\n      count += 1\n  return count\n')

In [42]:
normalize(base_output) == normalize(instruct_output)

True

In [24]:
def _truncated(suffix: str):
    s = ""
    for u in suffix:
        if u.isspace() and u != " ":
            break
        else:
            s += u
    return s

def _locate_char_in_string(full_output, suffix, verbose = False):
    try:
        idx = full_output.index(suffix)
    except:
        idx = full_output.index(_truncated(suffix))
    if verbose:
        print(full_output[:idx])

    return idx

_locate_char_in_string(alt_base_code, 'str(b)', True)

def tup_string(b):
    return 


30

In [25]:
def get_token_pos(tokenizer, input_text, char_idx, verbose = False):
    enc = tokenizer(
        input_text,
        return_offsets_mapping = True,
        add_special_tokens = False,
    )

    tokens = enc["input_ids"]
    offsets = enc["offset_mapping"]
    r_enc, r_dec, r_pos = None, None, None
    for tok_idx, (start, end) in enumerate(offsets):
        if start <= char_idx < end:
            if verbose:
                print("Token index:", tok_idx)
                print("Token text:", tokenizer.decode([tokens[tok_idx]]))
                print("Token offsets:", (start, end))
            r_pos = tok_idx
            r_enc = tokens[tok_idx]
            r_dec = tokenizer.decode([tokens[tok_idx]])
            break
    
    return tokens, r_pos, r_enc, r_dec 

In [26]:
from transformers import AutoTokenizer

gemma2_2b_tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")
gemma2_2b_it_tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")

char_index = _locate_char_in_string(alt_base_code, 'str(b)', True)
get_token_pos(gemma2_2b_tokenizer, alt_base_code, char_index, True)

def tup_string(b):
    return 
Token index: 10
Token text:  str
Token offsets: (29, 33)


([1293,
  117893,
  235298,
  1973,
  235278,
  235268,
  1245,
  108,
  141,
  773,
  1295,
  235278,
  235268,
  235275],
 10,
 1295,
 ' str')

In [27]:
def get_position_info(entry, verbose = False):
    try:
        ### get the outputs and task description params.
        instruct_output = entry["instruct_code"]
        base_output = entry["model_output"]
        task = entry["prompt"]
        test_list = entry["test_list"]

        ### create input prompt prefix based on templates used earlier.
        base_prefix = create_input_prompt_prefix(task, test_list, mode = "base")
        instruct_prefix = create_input_prompt_prefix(task, test_list, mode = "instruct")
        full_base_output = base_prefix + base_output
        full_instruct_output = instruct_prefix + instruct_output

        if verbose:
            print("Full base output", full_base_output)
            print("Full instruct output", full_instruct_output)

        ### extract only the code part and get the first notable difference suffix based on normalized AST diffs.
        base_code = _extract_code_part(base_prefix, base_output)
        instruct_code = _extract_code_part(instruct_prefix, instruct_output)
        diff_idx, diff_base_suffix, diff_instruct_suffix = find_first_difference(base_code, instruct_code, verbose = verbose)

        if verbose:
            print("Base suffix", diff_base_suffix)
            print("Instruct suffix", diff_instruct_suffix)

        ### locate the suffix in the full output, to be used later.
        base_locate_idx = _locate_char_in_string(full_base_output, diff_base_suffix, verbose = verbose)
        instruct_locate_idx = _locate_char_in_string(full_instruct_output, diff_instruct_suffix, verbose = verbose)

        if verbose:
            print("Base locate idx", base_locate_idx)
            print("Instruct locate idx", instruct_locate_idx)

        ### get token_pos of the first notable difference.
        all_base_tokens, base_token_pos, base_token_enc, base_token_dec = get_token_pos(gemma2_2b_tokenizer, full_base_output, base_locate_idx, verbose = verbose)
        all_instruct_tokens, instruct_token_pos, instruct_token_enc, instruct_token_dec = get_token_pos(gemma2_2b_it_tokenizer, full_instruct_output, instruct_locate_idx, verbose = verbose)

        return {
            "base_token_pos": base_token_pos,
            "base_token_enc": base_token_enc,
            "base_token_dec": base_token_dec,
            "instruct_token_pos": instruct_token_pos,
            "instruct_token_enc": instruct_token_enc,
            "instruct_token_dec": instruct_token_dec
        }
    
    except Exception as e:
        print("Exception", e)
        return None

In [28]:
sample = different_answer_list[0]

In [29]:
sample

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 3,
 'prompt': 'Write a python function to identify non-prime numbers.',
 'code': 'import math\ndef is_not_prime(n):\n    result = False\n    for i in range(2,int(math.sqrt(n)) + 1):\n        if n % i == 0:\n            result = True\n    return result',
 'test_imports': [],
 'test_list': ['assert is_not_prime(2) == False',
  'assert is_not_prime(10) == True',
  'assert is_not_prime(35) == True',
  'assert is_not_prime(37) == False'],
 'model_output': 'def is_not_prime(number):\n    if number == 2:\n        return False\n    if number == 1:\n        return True\n    if number % 2 == 0:\n        return False\n    for i in range(3, number, 2):\n        if number % i == 0:\n            return False\n    return True',
 'instruct_code': 'def is_not_prime(n):\n    if n <= 1:\n        return True\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return True\n    return False\n'}

In [30]:
full_base_output = """You are an expert Python programmer, and here is your task: Write a python function to identify non-prime numbers. Your code should pass these tests:

assert is_not_prime(2) == False
assert is_not_prime(10) == True
assert is_not_prime(35) == True
assert is_not_prime(37) == False
Write your code below starting with "```python" and ending with "```".
```python
def is_not_prime(number):
    if number == 2:
        return False
    if number == 1:
        return True
    if number % 2 == 0:
        return False
    for i in range(3, number, 2):
        if number % i == 0:
            return False
    return True"""

base_suffix = """umber == 2:"""

full_base_output.index(base_suffix)

395

In [31]:
get_token_pos(gemma2_2b_tokenizer, full_base_output, 395, True)

Token index: 115
Token text:  number
Token offsets: (393, 400)


([2045,
  708,
  671,
  13865,
  21237,
  70288,
  235269,
  578,
  1517,
  603,
  861,
  6911,
  235292,
  15615,
  476,
  17706,
  1411,
  577,
  11441,
  2173,
  235290,
  5773,
  5968,
  235265,
  3883,
  3409,
  1412,
  2307,
  1450,
  7782,
  235292,
  109,
  2994,
  603,
  235298,
  1665,
  235298,
  5773,
  235278,
  235284,
  235275,
  1159,
  7662,
  108,
  2994,
  603,
  235298,
  1665,
  235298,
  5773,
  235278,
  235274,
  235276,
  235275,
  1159,
  5569,
  108,
  2994,
  603,
  235298,
  1665,
  235298,
  5773,
  235278,
  235304,
  235308,
  235275,
  1159,
  5569,
  108,
  2994,
  603,
  235298,
  1665,
  235298,
  5773,
  235278,
  235304,
  235324,
  235275,
  1159,
  7662,
  108,
  5559,
  861,
  3409,
  3582,
  8035,
  675,
  664,
  1917,
  7774,
  235281,
  578,
  16851,
  675,
  664,
  1917,
  2776,
  108,
  1917,
  7774,
  108,
  1293,
  603,
  235298,
  1665,
  235298,
  5773,
  235278,
  4308,
  1245,
  108,
  141,
  648,
  1758,
  1159,
  235248,
  235284,
 

In [32]:
get_position_info(different_answer_list[0], verbose = True)

Full base output You are an expert Python programmer, and here is your task: Write a python function to identify non-prime numbers. Your code should pass these tests:

assert is_not_prime(2) == False
assert is_not_prime(10) == True
assert is_not_prime(35) == True
assert is_not_prime(37) == False
Write your code below starting with "```python" and ending with "```".
```python
def is_not_prime(number):
    if number == 2:
        return False
    if number == 1:
        return True
    if number % 2 == 0:
        return False
    for i in range(3, number, 2):
        if number % i == 0:
            return False
    return True
Full instruct output You are an expert Python programmer, and here is your task: Write a python function to identify non-prime numbers. Your code should pass these tests:

assert is_not_prime(2) == False
assert is_not_prime(10) == True
assert is_not_prime(35) == True
assert is_not_prime(37) == False
Write your code, without docstrings, below starting with "```pytho

{'base_token_pos': 115,
 'base_token_enc': 1758,
 'base_token_dec': ' number',
 'instruct_token_pos': 121,
 'instruct_token_enc': 5718,
 'instruct_token_dec': ' <='}

In [33]:
get_position_info(different_answer_list[1])

{'base_token_pos': 133,
 'base_token_enc': 773,
 'base_token_dec': 'return',
 'instruct_token_pos': 138,
 'instruct_token_enc': 746,
 'instruct_token_dec': 'for'}

In [34]:
different_answer_list[2]

{'source_file': "Mike's Copy of Benchmark Questions Verification V2.ipynb",
 'task_id': 56,
 'prompt': 'Write a python function to check if a given number is one less than twice its reverse.',
 'code': 'def rev(num):    \n    rev_num = 0\n    while (num > 0):  \n        rev_num = (rev_num * 10 + num % 10) \n        num = num // 10  \n    return rev_num  \ndef check(n):    \n    return (2 * rev(n) == n + 1)  ',
 'test_imports': [],
 'test_list': ['assert check(70) == False',
  'assert check(23) == False',
  'assert check(73) == True'],
 'model_output': 'def check(n):\n    if n == 0:\n        return False\n    if n < 0:\n        n = -n\n    rev = 0\n    while n > 0:\n        rev = rev * 10 + n % 10\n        n = n // 10\n    if n == 0:\n        return True\n    if n == rev - 1:\n        return True\n    return False',
 'instruct_code': 'def check(n):\n  return n == 2 * int(str(n)[::-1]) - 1\n'}

In [35]:
get_position_info(different_answer_list[2])

{'base_token_pos': 93,
 'base_token_enc': 648,
 'base_token_dec': 'if',
 'instruct_token_pos': 98,
 'instruct_token_enc': 773,
 'instruct_token_dec': 'return'}

**it is observed that the actual token_pos differs from the calculated token_pos by +1, so we should add +1 to the calculated token pos for a more fair assessment.**

- see prompt_0.
- see prompt_1.
- see prompt_2.

In [36]:
def get_position_info_corrected(entry):
    position_info = get_position_info(entry)
    if position_info is None:
        return entry
    else:
        position_info["base_token_pos"] += 1
        position_info["instruct_token_pos"] += 1
    
    entry["position_info"] = position_info
    return entry

In [37]:
for idx, entry in enumerate(different_answer_list):
    entry = get_position_info_corrected(entry)
    if "position_info" not in entry:
        print("Issue with task and idx", entry["task_id"], idx)

Exception '(' was never closed (<unknown>, line 27)
Issue with task and idx 160 10


In [38]:
with open("../outputs/selected_with_position_info.json", "w") as f:
    json.dump(different_answer_list, f, indent = 4)